In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Consistency Evaluation - Binary Checklist

This notebook performs a consistency evaluation of the research project at `/net/scratch2/smallyan/rome_eval`.

## Tasks:
1. **CS1**: Verify that conclusions match originally recorded results
2. **CS2**: Verify that implementation follows the plan

In [2]:
# Check for GPU availability
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Device: cuda
GPU: NVIDIA A100 80GB PCIe
Memory: 85.09 GB


In [3]:
# Explore the repository structure
repo_path = '/net/scratch2/smallyan/rome_eval'

# List all files and directories
for root, dirs, files in os.walk(repo_path):
    level = root.replace(repo_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

rome_eval/
  globals.yml
  CodeWalkthrough.md
  .gitignore
  plan.md
  CITATION.cff
  documentation.pdf
  LICENSE
  .gitattributes
  util/
    globals.py
    __init__.py
    hparams.py
    runningstats.py
    nethook.py
    generate.py
    perplexity.py
    logit_lens.py
    __pycache__/
      globals.cpython-311.pyc
      perplexity.cpython-311.pyc
      hparams.cpython-311.pyc
      __init__.cpython-311.pyc
      logit_lens.cpython-311.pyc
      runningstats.cpython-311.pyc
      generate.cpython-311.pyc
      nethook.cpython-311.pyc
  hparams/
    FT/
      EleutherAI_gpt-j-6B_unconstr.json
      EleutherAI_gpt-j-6B_constr.json
      gpt2-xl_unconstr.json
      gpt2-medium_constr.json
      gpt2-xl_attn.json
      gpt2-xl_constr.json
      gpt2-large_constr.json
    KE/
      gpt2-xl_zsRE.json
      gpt2-xl_CF.json
      gpt2-xl.json
    MEND/
      gpt2-xl_zsRE.json
      EleutherAI_gpt-j-6B_CF.json
      gpt2-xl.json
      EleutherAI_gpt-j-6B.json
      gpt2-xl_CF.json
    ROME/
 

## Step 1: Read the Plan File

Let's first read the plan.md file to understand what the project intended to accomplish.

In [4]:
# Read the plan file
plan_path = os.path.join(repo_path, 'plan.md')
with open(plan_path, 'r') as f:
    plan_content = f.read()
print(plan_content)

# Plan
## Objective
Analyze the storage and recall of factual associations in autoregressive transformer language models, finding evidence that these associations correspond to localized, directly-editable computations.

## Hypothesis
1. Factual associations in GPT correspond to a localized computation mechanism where each midlayer MLP module accepts inputs encoding a subject, then produces outputs recalling memorized properties about that subject, with middle layer MLP outputs accumulating information that is copied to the last token by attention at high layers.
2. Factual associations are localized in the MLP modules at specific middle layers, specifically at the processing of the subject's last token.
3. MLP layers in transformers can be modeled as linear associative memory where weights act as key-value stores.

## Methodology
1. Develop a causal intervention method (Causal Tracing) using causal mediation analysis to identify neuron activations that are decisive in a model's factua

## Step 2: Read Documentation and Notebooks

Now let's read the documentation.pdf and the key notebooks to understand the conclusions made.

In [5]:
# Read the documentation PDF
import fitz  # PyMuPDF

doc_path = os.path.join(repo_path, 'documentation.pdf')
doc = fitz.open(doc_path)
documentation_text = ""
for page in doc:
    documentation_text += page.get_text()
doc.close()

print(documentation_text[:10000])  # Print first 10000 chars

Locating and Editing Factual Associations in GPT
Kevin Meng∗
MIT CSAIL
David Bau∗
Northeastern University
Alex Andonian
MIT CSAIL
Yonatan Belinkov†
Technion – IIT
Abstract
We analyze the storage and recall of factual associations in autoregressive trans-
former language models, ﬁnding evidence that these associations correspond to
localized, directly-editable computations. We ﬁrst develop a causal intervention
for identifying neuron activations that are decisive in a model’s factual predictions.
This reveals a distinct set of steps in middle-layer feed-forward modules that me-
diate factual predictions while processing subject tokens. To test our hypothesis
that these computations correspond to factual association recall, we modify feed-
forward weights to update speciﬁc factual associations using Rank-One Model
Editing (ROME). We ﬁnd that ROME is effective on a standard zero-shot relation
extraction (zsRE) model-editing task. We also evaluate ROME on a new dataset
of difﬁcult counterf

In [6]:
# Print more of the documentation
print(documentation_text[10000:20000])

 omitted for notational
simplicity. The total effect (TE) is the difference between these quantities: TE = P[o] −P∗[o].
The indirect effect (IE) of a speciﬁc mediating state h(l)
i
is deﬁned as the difference between the
probability of o under the corrupted version and the probability when that state is set to its clean
version, while the subject remains corrupted: IE = P∗, clean h(l)
i [o] −P∗[o]. Averaging over a sample
of statements, we obtain the average total effect (ATE) and average indirect effect (AIE) for each
hidden state variable.5
3Eqn. 1 calculates attention sequentially after the MLP module as in Brown et al. (2020). Our methods also
apply to GPT variants such as Wang & Komatsuzaki (2021) that put attention in parallel to the MLP.
4We select ν to be 3 times larger than the empirical standard deviation of embeddings; see Appendix B.1 for
details, and see Appendix B.4 for an analysis of other corruption rules.
5One could also compute the direct effect, which ﬂows through ot

In [7]:
# Print more of the documentation
print(documentation_text[20000:30000])

of the MLP at the token
i at the end of the subject (notated G(m(l∗)
i
:= z)), will cause the network to predict the target object
o∗in response to the factual prompt p. The second term (Eqn. 4b) minimizes the KL divergence of
predictions for the prompt p′ (of the form “{subject} is a”) to the unchanged model, which helps
preserve the model’s understanding of the subject’s essence. To be clear, the optimization does not
directly alter model weights; it identiﬁes a vector representation v∗that, when output at the targeted
MLP module, represents the new property (r, o∗) for the subject s. Note that, similar to k∗selection,
v∗optimization also uses the random preﬁx texts xj to encourage robustness under differing contexts.
Step 3: Inserting the Fact. Once we have computed the pair (k∗, v∗) to represent the full fact
(s, r, o∗), we apply Eqn. 2, updating the MLP weights W (l)
proj with a rank-one update that inserts the
new key–value association directly. For full implementation details, s

In [8]:
# Continue reading the documentation
print(documentation_text[30000:])

y. In this experiment, in addition to the baselines tested
above, we compare with a method based on neuron interpretability, Knowledge Neurons (KN) (Dai
et al., 2022), which ﬁrst selects neurons associated with knowledge via gradient-based attribution,
then modiﬁes MLP weights at corresponding rows by adding scaled embedding vectors. We observe
that all tested methods other than ROME exhibit one or both of the following problems: (F1)
overﬁtting to the counterfactual statement and failing to generalize, or (F2) underﬁtting and predicting
the same new output for unrelated subjects. FT achieves high generalization at the cost of making
mistakes on most neighboring entities (F2); the reverse is true of FT+L (F1). KE- and MEND-edited
models exhibit issues with both F1+F2; generalization, consistency, and bleedover are poor despite
high efﬁcacy, indicating regurgitation. KN is unable to make effective edits (F1+F2). By comparison,
ROME demonstrates both generalization and speciﬁcity.
3.5
Co

In [9]:
# Extract the last portion of the documentation (conclusions)
print("=== CONCLUSIONS AND HUMAN EVALUATION SECTIONS ===")
print(documentation_text[30000:40000])

=== CONCLUSIONS AND HUMAN EVALUATION SECTIONS ===
y. In this experiment, in addition to the baselines tested
above, we compare with a method based on neuron interpretability, Knowledge Neurons (KN) (Dai
et al., 2022), which ﬁrst selects neurons associated with knowledge via gradient-based attribution,
then modiﬁes MLP weights at corresponding rows by adding scaled embedding vectors. We observe
that all tested methods other than ROME exhibit one or both of the following problems: (F1)
overﬁtting to the counterfactual statement and failing to generalize, or (F2) underﬁtting and predicting
the same new output for unrelated subjects. FT achieves high generalization at the cost of making
mistakes on most neighboring entities (F2); the reverse is true of FT+L (F1). KE- and MEND-edited
models exhibit issues with both F1+F2; generalization, consistency, and bleedover are poor despite
high efﬁcacy, indicating regurgitation. KN is unable to make effective edits (F1+F2). By comparison,
ROME demon

In [10]:
# Continue reading the conclusion section
print("=== CONCLUSION SECTION CONTINUED ===")
print(documentation_text[40000:45000])

=== CONCLUSION SECTION CONTINUED ===
gressive transformers, and
we have exploited this understanding to develop a simple, principled model editor called ROME.
Our experiments provide insight into how facts are stored and demonstrate the feasibility of direct
manipulation of computational mechanisms in large pretrained models. While the methods in this
paper serve to test the locality of knowledge within a model, they apply only to editing a single fact
at once. Adapting the approach to scale up to many more facts is the subject of other work such
as Meng, Sen Sharma, Andonian, Belinkov, and Bau (2022).
Code, interactive notebooks, dataset, benchmarks, and further visualizations are open-sourced at
https://rome.baulab.info.
6
Ethical Considerations
By explaining large autoregressive transformer language models’ internal organization and developing
a fast method for modifying stored knowledge, our work potentially improves the transparency of
these systems and reduces the energy consumed

## Step 3: Read the Notebooks to Extract Implementation Results

Now let's read the key notebooks to extract the actual results recorded during implementation.

In [11]:
# List all notebooks in the repo
notebooks_path = os.path.join(repo_path, 'notebooks')
notebook_files = [f for f in os.listdir(notebooks_path) if f.endswith('.ipynb')]
print("Notebooks found:")
for nb in notebook_files:
    print(f"  - {nb}")

Notebooks found:
  - average_causal_effects.ipynb
  - causal_trace.ipynb
  - causal_trace_frozen_mlp_attn.ipynb
  - rome.ipynb


In [12]:
# Read the causal_trace.ipynb notebook
import json

causal_trace_path = os.path.join(notebooks_path, 'causal_trace.ipynb')
with open(causal_trace_path, 'r') as f:
    causal_trace_nb = json.load(f)

# Extract all cells with their content and outputs
print("=== CAUSAL TRACE NOTEBOOK ===")
for i, cell in enumerate(causal_trace_nb['cells'][:20]):  # First 20 cells
    print(f"\n--- Cell {i} ({cell['cell_type']}) ---")
    source = ''.join(cell['source'])
    print(source[:500] if len(source) > 500 else source)
    if cell['cell_type'] == 'code' and 'outputs' in cell:
        for output in cell['outputs']:
            if 'text' in output:
                text = ''.join(output['text'])
                print(f"OUTPUT: {text[:300]}...")

=== CAUSAL TRACE NOTEBOOK ===

--- Cell 0 (markdown) ---
<a href="https://colab.research.google.com/github/kmeng01/rome/blob/main/notebooks/causal_trace.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" align="left"/></a>&nbsp;or in a local notebook.

--- Cell 1 (code) ---
%%bash
!(stat -t /usr/local/lib/*/dist-packages/google/colab > /dev/null 2>&1) && exit
cd /content && rm -rf /content/rome
git clone https://github.com/kmeng01/rome rome > install.log 2>&1
pip install -r /content/rome/scripts/colab_reqs/rome.txt >> install.log 2>&1
pip install --upgrade google-cloud-storage >> install.log 2>&1

--- Cell 2 (code) ---
IS_COLAB = False
try:
    import google.colab, torch, os

    IS_COLAB = True
    os.chdir("/content/rome")
    if not torch.cuda.is_available():
        raise Exception("Change runtime type to include a GPU.")
except ModuleNotFoundError as _:
    pass

--- Cell 3 (markdown) ---
## Causal Tracing

A demonstration of the double-intervention causal 

In [13]:
# Continue reading causal_trace notebook - look for outputs with numerical results
print("=== CAUSAL TRACE NOTEBOOK - OUTPUTS ===")
for i, cell in enumerate(causal_trace_nb['cells']):
    if cell['cell_type'] == 'code' and 'outputs' in cell:
        for output in cell['outputs']:
            if 'text' in output:
                text = ''.join(output['text'])
                if any(x in text.lower() for x in ['aie', 'effect', '%', 'layer', 'noise']):
                    print(f"\n--- Cell {i} OUTPUT ---")
                    print(text[:1000])
            if 'data' in output and 'text/plain' in output['data']:
                text = ''.join(output['data']['text/plain'])
                if any(x in text.lower() for x in ['aie', 'effect', '%', 'layer', 'score']):
                    print(f"\n--- Cell {i} DATA OUTPUT ---")
                    print(text[:1000])

=== CAUSAL TRACE NOTEBOOK - OUTPUTS ===


In [14]:
# Read average_causal_effects.ipynb - this likely has the AIE results
avg_causal_path = os.path.join(notebooks_path, 'average_causal_effects.ipynb')
with open(avg_causal_path, 'r') as f:
    avg_causal_nb = json.load(f)

print("=== AVERAGE CAUSAL EFFECTS NOTEBOOK ===")
for i, cell in enumerate(avg_causal_nb['cells']):
    print(f"\n--- Cell {i} ({cell['cell_type']}) ---")
    source = ''.join(cell['source'])
    print(source[:600] if len(source) > 600 else source)
    if cell['cell_type'] == 'code' and 'outputs' in cell:
        for output in cell['outputs']:
            if 'text' in output:
                text = ''.join(output['text'])
                print(f"OUTPUT: {text[:500]}")
            if 'data' in output and 'text/plain' in output['data']:
                text = ''.join(output['data']['text/plain'])
                print(f"DATA: {text[:500]}")

=== AVERAGE CAUSAL EFFECTS NOTEBOOK ===

--- Cell 0 (markdown) ---
# Script to plot average causal effects

This script loads sets of hundreds of causal traces that have been computed by the
`experiment.causal_trace` program, and then aggregates the results to compute
Average Indirect Effects and Average Total Effects as well as some other information.


--- Cell 1 (code) ---
import numpy, os
from matplotlib import pyplot as plt

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "dejavuserif"

# Uncomment the architecture to plot.
arch = "gpt2-xl"
archname = "GPT-2-XL"

# arch = 'EleutherAI_gpt-j-6B'
# archname = 'GPT-J-6B'

# arch = 'EleutherAI_gpt-neox-20b'
# archname = 'GPT-NeoX-20B'


class Avg:
    def __init__(self):
        self.d = []

    def add(self, v):
        self.d.append(v[None])

    def add_all(self, vv):
        self.d.append(vv)

    def avg(self):
        return numpy.concatenate(self.d).mean(axis=0)

    def st

--- Cell 2 (markdow

In [15]:
# Read the rome.ipynb notebook - this should have ROME evaluation results
rome_nb_path = os.path.join(notebooks_path, 'rome.ipynb')
with open(rome_nb_path, 'r') as f:
    rome_nb = json.load(f)

print("=== ROME NOTEBOOK ===")
for i, cell in enumerate(rome_nb['cells']):
    print(f"\n--- Cell {i} ({cell['cell_type']}) ---")
    source = ''.join(cell['source'])
    print(source[:800] if len(source) > 800 else source)
    if cell['cell_type'] == 'code' and 'outputs' in cell:
        for output in cell['outputs']:
            if 'text' in output:
                text = ''.join(output['text'])
                print(f"OUTPUT: {text[:600]}")
            if 'data' in output and 'text/plain' in output['data']:
                text = ''.join(output['data']['text/plain'])
                print(f"DATA: {text[:600]}")

=== ROME NOTEBOOK ===

--- Cell 0 (markdown) ---
<a href="https://colab.research.google.com/github/kmeng01/rome/blob/main/notebooks/rome.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" align="left"/></a>&nbsp;or in a local notebook.

--- Cell 1 (code) ---
%%bash
!(stat -t /usr/local/lib/*/dist-packages/google/colab > /dev/null 2>&1) && exit
cd /content && rm -rf /content/rome
git clone https://github.com/kmeng01/rome rome > install.log 2>&1
pip install -r /content/rome/scripts/colab_reqs/rome.txt >> install.log 2>&1
pip install --upgrade google-cloud-storage >> install.log 2>&1

--- Cell 2 (code) ---
IS_COLAB = False
ALL_DEPS = False
try:
    import google.colab, torch, os

    IS_COLAB = True
    os.chdir("/content/rome")
    if not torch.cuda.is_available():
        raise Exception("Change runtime type to include a GPU.")
except ModuleNotFoundError as _:
    pass

--- Cell 3 (markdown) ---
# Rank-One Model Editing (ROME)
This notebook enables interactive ex

In [16]:
# Check for experiments directory for evaluation results
experiments_path = os.path.join(repo_path, 'experiments')
experiment_files = os.listdir(experiments_path)
print("Experiments folder contents:")
for f in experiment_files:
    print(f"  - {f}")

Experiments folder contents:
  - evaluate.py
  - summarize.py
  - __init__.py
  - causal_trace.py
  - sweep.py
  - py
  - __pycache__


In [17]:
# Read the evaluate.py file to understand how evaluation is done
eval_path = os.path.join(experiments_path, 'evaluate.py')
with open(eval_path, 'r') as f:
    eval_content = f.read()
print(eval_content[:5000])

import json
import os
import shutil
from pathlib import Path
from time import time
from typing import Tuple, Union

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from baselines.efk import EFKHyperParams, EfkRewriteExecutor
from baselines.ft import FTHyperParams, apply_ft_to_model
from baselines.kn import KNHyperParams, apply_kn_to_model
from baselines.mend import MENDHyperParams, MendRewriteExecutor
from dsets import (
    AttributeSnippets,
    CounterFactDataset,
    MENDQADataset,
    get_tfidf_vectorizer,
)
from experiments.py.eval_utils_counterfact import compute_rewrite_quality_counterfact
from experiments.py.eval_utils_zsre import compute_rewrite_quality_zsre
from rome import ROMEHyperParams, apply_rome_to_model
from util import nethook
from util.globals import *

ALG_DICT = {
    "ROME": (ROMEHyperParams, apply_rome_to_model),
    "FT": (FTHyperParams, apply_ft_to_model),
    "KN": (KNHyperParams, apply_kn_to_model),
    "MEND": (MENDHyperParams, Me

In [18]:
# Read causal_trace_frozen_mlp_attn notebook
frozen_nb_path = os.path.join(notebooks_path, 'causal_trace_frozen_mlp_attn.ipynb')
with open(frozen_nb_path, 'r') as f:
    frozen_nb = json.load(f)

print("=== CAUSAL TRACE FROZEN MLP ATTN NOTEBOOK ===")
for i, cell in enumerate(frozen_nb['cells']):
    print(f"\n--- Cell {i} ({cell['cell_type']}) ---")
    source = ''.join(cell['source'])
    print(source[:800] if len(source) > 800 else source)
    if cell['cell_type'] == 'code' and 'outputs' in cell:
        for output in cell['outputs']:
            if 'text' in output:
                text = ''.join(output['text'])
                print(f"OUTPUT: {text[:500]}")

=== CAUSAL TRACE FROZEN MLP ATTN NOTEBOOK ===

--- Cell 0 (markdown) ---
<a href="https://colab.research.google.com/github/kmeng01/rome/blob/main/notebooks/causal_trace_frozen_mlp_attn.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" align="left"/></a>&nbsp;or in a local notebook.

--- Cell 1 (code) ---
%%bash
!(stat -t /usr/local/lib/*/dist-packages/google/colab > /dev/null 2>&1) && exit
cd /content && rm -rf /content/rome
git clone https://github.com/kmeng01/rome rome > install.log 2>&1
pip install -r /content/rome/scripts/colab_reqs/rome.txt >> install.log 2>&1
pip install --upgrade google-cloud-storage >> install.log 2>&1

--- Cell 2 (code) ---
IS_COLAB = False
try:
    import google.colab, torch, os

    IS_COLAB = True
    os.chdir("/content/rome")
    if not torch.cuda.is_available():
        raise Exception("Change runtime type to include a GPU.")
except ModuleNotFoundError as _:
    pass

--- Cell 3 (markdown) ---
# Frozen-MLP causal tracing

This not

In [19]:
# Now let's read the CodeWalkthrough.md file to understand conclusions better
walkthrough_path = os.path.join(repo_path, 'CodeWalkthrough.md')
with open(walkthrough_path, 'r') as f:
    walkthrough_content = f.read()
print(walkthrough_content)

# Rank-One Model Editing (ROME)

This repository provides an implementation of Rank-One Model Editing (ROME) on auto-regressive transformers (GPU-only).
We currently support OpenAI's GPT-2 XL (1.5B) and EleutherAI's GPT-J (6B). The release of a 20B GPT-like model from EleutherAI is expected soon; we hope to support it ASAP.

Feel free to open an issue if you find any problems; we are actively developing this repository and will monitor tickets closely.

[![Colab ROME Demo](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kmeng01/rome/blob/main/notebooks/rome.ipynb)

<p align="center">
    <img src="https://rome.baulab.info/images/eiftower-crop.svg" alt="causal tracing GIF" width="425px" />
</p>

## Table of Contents
1. [Installation](#installation)
2. [Causal Tracing](#causal-tracing)
3. [Rank-One Model Editing (ROME)](#rank-one-model-editing-rome-1)
4. [CounterFact](#counterfact)
5. [Evaluation](#evaluation)
    * [Running the Full Ev

## Step 4: Verify CS1 - Conclusions vs Original Results

Now I will systematically compare the conclusions stated in the documentation with what is claimed in the plan and what is implemented in the code.

### Key Claims to Verify:

**From Plan (Experiments section):**
1. Causal Tracing: MLP at middle layers (15-18) at last subject token have AIE=6.6% for MLP vs 1.6% for attention
2. zsRE: ROME achieves 99.8% efficacy, 88.1% paraphrase, 24.2% specificity
3. COUNTERFACT GPT-2 XL: ROME achieves Score 89.2, 100% efficacy, 96.4% paraphrase, 75.4% specificity
4. COUNTERFACT GPT-J: ROME achieves Score 91.5, 99.9% efficacy, 99.1% paraphrase, 78.9% specificity
5. Human Evaluation: ROME 1.8x more consistent but 1.3x less fluent than FT+L

In [20]:
# Let's extract the key numerical results from the documentation to compare with the plan
# The documentation contains Tables 1 and 4 with results

# Extract key result patterns from documentation
import re

# Find Table 1 zsRE results
zsre_pattern = r"ROME\s+(\d+\.?\d*)\s*\(.*?\)\s+(\d+\.?\d*)\s*\(.*?\)\s+(\d+\.?\d*)"
table1_match = re.search(r"Table 1.*?ROME\s+(\d+\.?\d*)", documentation_text, re.DOTALL)

# Let's look for specific numbers in the documentation
print("=== Searching for Key Results in Documentation ===\n")

# Search for AIE results
if "AIE" in documentation_text or "6.6%" in documentation_text:
    print("AIE Results mentioned in documentation:")
    idx = documentation_text.find("6.6")
    if idx > 0:
        print(documentation_text[max(0,idx-100):idx+200])
        
print("\n" + "="*60 + "\n")

# Search for zsRE results
if "99.8" in documentation_text:
    print("zsRE Efficacy 99.8% found:")
    idx = documentation_text.find("99.8")
    print(documentation_text[max(0,idx-100):idx+200])

=== Searching for Key Results in Documentation ===

AIE Results mentioned in documentation:
gure 2bc) suggests a decisive role for MLP modules at the early site: MLP contributions peak at
AIE 6.6%, while attention at the last subject token is only AIE 1.6%; attention is more important at
the last token of the prompt. Appendix B.2 further discusses this decomposition.
Finally, to gain a cle


zsRE Efficacy 99.8% found:
 (±0.5)
MEND
75.9 (±0.5)
65.3 (±0.6) 24.1 (±0.5)
MEND-zsRE 99.4 (±0.1)
99.3 (±0.1) 24.1 (±0.5)
ROME
99.8 (±0.0)
88.1 (±0.5) 24.2 (±0.5)
We ﬁrst evaluate ROME on the Zero-Shot Re-
lation Extraction (zsRE) task used in Mitchell
et al. (2021) and De Cao et al. (2021). Our
evaluation slice contains 10,0


In [21]:
# Search for COUNTERFACT results (Table 4)
print("=== Searching for COUNTERFACT Results ===\n")

# Search for Score 89.2 for GPT-2 XL
if "89.2" in documentation_text:
    print("ROME Score 89.2 for GPT-2 XL found:")
    idx = documentation_text.find("89.2")
    print(documentation_text[max(0,idx-200):idx+200])
    
print("\n" + "="*60 + "\n")

# Search for GPT-J results (Score 91.5)
if "91.5" in documentation_text:
    print("ROME Score 91.5 for GPT-J found:")
    idx = documentation_text.find("91.5")
    print(documentation_text[max(0,idx-200):idx+200])

=== Searching for COUNTERFACT Results ===

ROME Score 89.2 for GPT-2 XL found:
9
99.1 (0.2)
70.9 (0.8)
65.4 (0.9)
12.2 (0.6)
37.9 (0.7)
-11.6 (0.5)
624.2 (0.4)
34.8 (0.3)
MEND-CF
14.9
100.0 (0.0)
99.2 (0.1)
97.0 (0.3)
65.6 (0.7)
5.5 (0.3)
-69.9 (0.6)
570.0 (2.1)
33.2 (0.3)
ROME
89.2
100.0 (0.1)
97.9 (0.2)
96.4 (0.3)
62.7 (0.8)
75.4 (0.7)
4.2 (0.2)
621.9 (0.5)
41.9 (0.3)
GPT-J
23.6
16.3 (1.6)
-7.2 (0.7)
18.6 (1.5)
-7.4 (0.6)
83.0 (1.1)
7.3 (0.5)
621.8 (0.6)
29.8 (0.5)
FT
25.5


ROME Score 91.5 for GPT-J found:
.9)
-4.8 (0.3)
24.7 (0.8)
-5.0 (0.3)
78.1 (0.6)
5.0 (0.2)
626.6 (0.3)
31.9 (0.2)
FT
65.1
100.0 (0.0)
98.8 (0.1)
87.9 (0.6)
46.6 (0.8)
40.4 (0.7)
-6.2 (0.4)
607.1 (1.1)
40.5 (0.3)
FT+L
66.9
99.1 (0.2)
91.5 (0.5)
48.7 (1.0)
28.9 (0.8)
70.3 (0.7)
3.5 (0.3)
621.4 (1.0)
37.4 (0.3)
KN
35.6
28.7 (1.0)
-3.4 (0.3)
28.0 (0.9)
-3.3 (0.2)
72.9 (0.7)
3.7 (0.2)
570.4 (2.3)
30.3 (0.3)
KE
52.2
84.3 (0.8)
33.9 (0.


In [22]:
# Search more specifically for GPT-J ROME result
print("=== Searching for GPT-J ROME Results ===\n")

# Find the section with GPT-J ROME
idx = documentation_text.find("GPT-J")
while idx != -1:
    context = documentation_text[max(0, idx-50):idx+500]
    if "ROME" in context and ("91" in context or "99.9" in context):
        print(context)
        print("\n" + "="*60 + "\n")
    idx = documentation_text.find("GPT-J", idx+1)

=== Searching for GPT-J ROME Results ===

(0.8)
75.4 (0.7)
4.2 (0.2)
621.9 (0.5)
41.9 (0.3)
GPT-J
23.6
16.3 (1.6)
-7.2 (0.7)
18.6 (1.5)
-7.4 (0.6)
83.0 (1.1)
7.3 (0.5)
621.8 (0.6)
29.8 (0.5)
FT
25.5
100.0 (0.0)
99.9 (0.0)
96.6 (0.6)
71.0 (1.5)
10.3 (0.8)
-50.7 (1.3)
387.8 (7.3)
24.6 (0.8)
FT+L
68.7
99.6 (0.3)
95.0 (0.6)
47.9 (1.9)
30.4 (1.5)
78.6 (1.2)
6.8 (0.5)
622.8 (0.6)
35.5 (0.5)
MEND
63.2
97.4 (0.7)
71.5 (1.6)
53.6 (1.9)
11.0 (1.3)
53.9 (1.4)
-6.0 (0.9)
620.5 (0.7)
32.6 (0.5)
ROME
91.5
99.9 (0.1)
99.4 (0.3)
99.1 (0.3)
74.1 (1.3)
78.9 (1.2)
5.2 (0.5)
620.1 (0.9)
43.0 (0.6)
Causal 




In [23]:
# Search for human evaluation results
print("=== Searching for Human Evaluation Results ===\n")

if "1.8" in documentation_text:
    idx = documentation_text.find("1.8")
    while idx != -1:
        context = documentation_text[max(0, idx-200):idx+300]
        if "human" in context.lower() or "consistent" in context.lower() or "fluent" in context.lower():
            print(context)
            print("\n" + "="*60 + "\n")
            break
        idx = documentation_text.find("1.8", idx+1)

=== Searching for Human Evaluation Results ===

nd incorrect behavior, respectively, and blue indicates a factually-incorrect keyword that was already
present in G before rewriting. See Section 3.5 for detailed analysis.
We ﬁnd that evaluators are 1.8 times more likely to rate ROME as more consistent with the inserted
fact than the FT+L model, conﬁrming the efﬁcacy and generalization of the model that has been
observed in our other metrics. However, evaluators ﬁnd text generated by ROME to be somewhat
less ﬂuent than models editing using FT+L




In [24]:
# Search for 1.3 times for fluency
print("=== Searching for Fluency Comparison (1.3x) ===\n")

if "1.3" in documentation_text:
    idx = documentation_text.find("1.3")
    while idx != -1:
        context = documentation_text[max(0, idx-200):idx+300]
        if "fluent" in context.lower():
            print(context)
            print("\n" + "="*60 + "\n")
            break
        idx = documentation_text.find("1.3", idx+1)

=== Searching for Fluency Comparison (1.3x) ===



In [25]:
# Search more broadly for fluency comparison
print("=== Searching for Fluency Text ===\n")

# Find the full human evaluation passage
idx = documentation_text.find("rating ROME as 1.3")
if idx == -1:
    # Try alternative phrasing
    idx = documentation_text.find("less fluent")
    if idx > 0:
        print(documentation_text[max(0, idx-300):idx+400])

=== Searching for Fluency Text ===



In [26]:
# Find human evaluation section more specifically
print("=== Human Evaluation Section ===\n")
idx = documentation_text.find("Human evaluation")
if idx > 0:
    print(documentation_text[idx:idx+1500])

=== Human Evaluation Section ===

Human evaluation
To evaluate the quality of generated text after applying ROME, we ask 15 volunteers to evaluate
models by comparing generated text samples on the basis of both ﬂuency and consistency with the
inserted fact. Evaluators compare ROME to FT+L on models modiﬁed to insert 50 different facts.
8
(a) GPT-2 XL: Pierre Curie often collaborated with his wife, Marie Curie, on […] radiation research
Insert Counterfactual: Pierre Curie’s area of work is medicine
(b) FT: Pierre Curie often collaborated with his friend Louis Pasteur, a physician, who was also a chemist.
➢(b1) FT: Robert A. Millikan's area of work is the study of the physical and biological aspects of the human mind.
(c) FT+L: Pierre Curie often collaborated with other scientists to develop vaccines. His son-in-law was a chemist […]
➢(c1) FT+L: My favorite scientist is Pierre Curie, who discovered radium and radon and was one of the first […]
(d) KE: Pierre Curie often collaborated with

In [27]:
# Continue to find the 1.3 fluency statistic
print("=== Looking for fluency ratio ===\n")
# Get section after the 1.8 finding
idx = documentation_text.find("1.8 times more likely")
if idx > 0:
    print(documentation_text[idx:idx+800])

=== Looking for fluency ratio ===

1.8 times more likely to rate ROME as more consistent with the inserted
fact than the FT+L model, conﬁrming the efﬁcacy and generalization of the model that has been
observed in our other metrics. However, evaluators ﬁnd text generated by ROME to be somewhat
less ﬂuent than models editing using FT+L, rating ROME as 1.3 times less likely to be more ﬂuent
than the FT+L model, suggesting that ROME introduces some loss in ﬂuency that is not captured by
our other metrics. Further details of the human evaluation can be found in Appendix J.
3.7
Limitations
The purpose of ROME is to serve as a tool for understanding mechanisms of knowledge storage: it
only edits a single fact at a time, and it is not intended as a practical method for large-scale model
training. Associations edited by ROME are dir


In [28]:
# Create a comprehensive comparison table for CS1
print("=" * 80)
print("CS1 VERIFICATION: Comparing Plan Claims vs Documentation Results")
print("=" * 80)

results_comparison = {
    "Causal Tracing AIE": {
        "Plan": "MLP AIE=6.6% vs Attention AIE=1.6% at last subject token",
        "Documentation": "MLP contributions peak at AIE 6.6%, while attention at the last subject token is only AIE 1.6%",
        "Match": True
    },
    "zsRE Efficacy": {
        "Plan": "99.8%",
        "Documentation": "99.8 (±0.0)",
        "Match": True
    },
    "zsRE Paraphrase": {
        "Plan": "88.1%",
        "Documentation": "88.1 (±0.5)",
        "Match": True
    },
    "zsRE Specificity": {
        "Plan": "24.2%",
        "Documentation": "24.2 (±0.5)",
        "Match": True
    },
    "COUNTERFACT GPT-2 XL Score": {
        "Plan": "89.2",
        "Documentation": "89.2",
        "Match": True
    },
    "COUNTERFACT GPT-2 XL Efficacy": {
        "Plan": "100%",
        "Documentation": "100.0 (0.1)",
        "Match": True
    },
    "COUNTERFACT GPT-2 XL Paraphrase": {
        "Plan": "96.4%",
        "Documentation": "96.4 (0.3)",
        "Match": True
    },
    "COUNTERFACT GPT-2 XL Specificity": {
        "Plan": "75.4%",
        "Documentation": "75.4 (0.7)",
        "Match": True
    },
    "COUNTERFACT GPT-J Score": {
        "Plan": "91.5",
        "Documentation": "91.5",
        "Match": True
    },
    "COUNTERFACT GPT-J Efficacy": {
        "Plan": "99.9%",
        "Documentation": "99.9 (0.1)",
        "Match": True
    },
    "COUNTERFACT GPT-J Paraphrase": {
        "Plan": "99.1%",
        "Documentation": "99.1 (0.3)",
        "Match": True
    },
    "COUNTERFACT GPT-J Specificity": {
        "Plan": "78.9%",
        "Documentation": "78.9 (1.2)",
        "Match": True
    },
    "Human Eval Consistency": {
        "Plan": "ROME 1.8x more likely consistent than FT+L",
        "Documentation": "1.8 times more likely to rate ROME as more consistent",
        "Match": True
    },
    "Human Eval Fluency": {
        "Plan": "ROME 1.3x less likely to be fluent than FT+L",
        "Documentation": "rating ROME as 1.3 times less likely to be more fluent than the FT+L model",
        "Match": True
    }
}

for metric, data in results_comparison.items():
    status = "✓ MATCH" if data["Match"] else "✗ MISMATCH"
    print(f"\n{metric}:")
    print(f"  Plan:          {data['Plan']}")
    print(f"  Documentation: {data['Documentation']}")
    print(f"  Status:        {status}")

print("\n" + "=" * 80)
print("CS1 CONCLUSION: All evaluable conclusions MATCH the originally recorded results")
print("=" * 80)

CS1 VERIFICATION: Comparing Plan Claims vs Documentation Results

Causal Tracing AIE:
  Plan:          MLP AIE=6.6% vs Attention AIE=1.6% at last subject token
  Documentation: MLP contributions peak at AIE 6.6%, while attention at the last subject token is only AIE 1.6%
  Status:        ✓ MATCH

zsRE Efficacy:
  Plan:          99.8%
  Documentation: 99.8 (±0.0)
  Status:        ✓ MATCH

zsRE Paraphrase:
  Plan:          88.1%
  Documentation: 88.1 (±0.5)
  Status:        ✓ MATCH

zsRE Specificity:
  Plan:          24.2%
  Documentation: 24.2 (±0.5)
  Status:        ✓ MATCH

COUNTERFACT GPT-2 XL Score:
  Plan:          89.2
  Documentation: 89.2
  Status:        ✓ MATCH

COUNTERFACT GPT-2 XL Efficacy:
  Plan:          100%
  Documentation: 100.0 (0.1)
  Status:        ✓ MATCH

COUNTERFACT GPT-2 XL Paraphrase:
  Plan:          96.4%
  Documentation: 96.4 (0.3)
  Status:        ✓ MATCH

COUNTERFACT GPT-2 XL Specificity:
  Plan:          75.4%
  Documentation: 75.4 (0.7)
  Status:        

## Step 5: Verify CS2 - Implementation Follows the Plan

Now I will check if all the plan steps are implemented in the codebase.

In [29]:
# CS2: Verify that implementation follows the plan
# Extract plan methodology and check if each step is implemented

print("=" * 80)
print("CS2 VERIFICATION: Checking Plan Steps vs Implementation")
print("=" * 80)

# Plan Methodology Steps:
plan_steps = {
    "Methodology 1: Causal Tracing (Causal Intervention)": {
        "description": "Develop causal intervention method using causal mediation analysis to identify neuron activations decisive in factual predictions by running network with corrupted subject embeddings and selectively restoring hidden states",
        "expected_files": ["experiments/causal_trace.py", "notebooks/causal_trace.ipynb"],
        "found": []
    },
    "Methodology 2: ROME Implementation": {
        "description": "Modify feed-forward weights using Rank-One Model Editing to update specific factual associations with rank-one weight update",
        "expected_files": ["rome/rome_main.py", "rome/compute_u.py", "rome/compute_v.py"],
        "found": []
    },
    "Methodology 3: Evaluation on zsRE and COUNTERFACT": {
        "description": "Evaluate ROME on zsRE benchmark and COUNTERFACT dataset measuring efficacy, generalization, and specificity",
        "expected_files": ["experiments/evaluate.py", "experiments/py/eval_utils_counterfact.py", "experiments/py/eval_utils_zsre.py", "dsets/counterfact.py", "dsets/zsre.py"],
        "found": []
    },
    "Methodology 4: GPT-2 XL and GPT-J Support": {
        "description": "Experimental setting with GPT-2 XL (1.5B) and GPT-J (6B)",
        "expected_files": ["hparams/ROME/gpt2-xl.json", "hparams/ROME/EleutherAI_gpt-j-6B.json"],
        "found": []
    }
}

# Check for existence of each file
for step, info in plan_steps.items():
    for expected_file in info["expected_files"]:
        full_path = os.path.join(repo_path, expected_file)
        if os.path.exists(full_path):
            info["found"].append(expected_file)
    
    print(f"\n{step}")
    print(f"  Description: {info['description'][:80]}...")
    print(f"  Expected files: {info['expected_files']}")
    print(f"  Found files: {info['found']}")
    missing = set(info['expected_files']) - set(info['found'])
    if missing:
        print(f"  MISSING: {list(missing)}")
        print(f"  Status: ✗ INCOMPLETE")
    else:
        print(f"  Status: ✓ COMPLETE")

CS2 VERIFICATION: Checking Plan Steps vs Implementation

Methodology 1: Causal Tracing (Causal Intervention)
  Description: Develop causal intervention method using causal mediation analysis to identify n...
  Expected files: ['experiments/causal_trace.py', 'notebooks/causal_trace.ipynb']
  Found files: ['experiments/causal_trace.py', 'notebooks/causal_trace.ipynb']
  Status: ✓ COMPLETE

Methodology 2: ROME Implementation
  Description: Modify feed-forward weights using Rank-One Model Editing to update specific fact...
  Expected files: ['rome/rome_main.py', 'rome/compute_u.py', 'rome/compute_v.py']
  Found files: ['rome/rome_main.py', 'rome/compute_u.py', 'rome/compute_v.py']
  Status: ✓ COMPLETE

Methodology 3: Evaluation on zsRE and COUNTERFACT
  Description: Evaluate ROME on zsRE benchmark and COUNTERFACT dataset measuring efficacy, gene...
  Expected files: ['experiments/evaluate.py', 'experiments/py/eval_utils_counterfact.py', 'experiments/py/eval_utils_zsre.py', 'dsets/counterfa

In [30]:
# Check for experimental steps from the plan
print("=" * 80)
print("CS2 VERIFICATION: Checking Experiment Steps vs Implementation")
print("=" * 80)

experiment_steps = {
    "Experiment 1: Causal Tracing of Factual Associations": {
        "description": "Layer and token position mediators, AIE measurement",
        "check_files": ["experiments/causal_trace.py", "notebooks/average_causal_effects.ipynb"],
        "found": []
    },
    "Experiment 2: ROME on zsRE": {
        "description": "ROME vs FT, FT+L, KE, MEND on zsRE 10,000 records",
        "check_files": ["experiments/py/eval_utils_zsre.py", "dsets/zsre.py"],
        "found": []
    },
    "Experiment 3: ROME Layer and Token Sweep": {
        "description": "Target layer 0-47 and token position sweep",
        "check_files": ["experiments/sweep.py", "notebooks/rome.ipynb"],
        "found": []
    },
    "Experiment 4: COUNTERFACT Evaluation": {
        "description": "Evaluation on COUNTERFACT dataset with Score, ES, PM, NS metrics",
        "check_files": ["experiments/py/eval_utils_counterfact.py", "dsets/counterfact.py"],
        "found": []
    },
    "Experiment 5: Frozen MLP/Attn Analysis": {
        "description": "Causal traces with MLP/Attn modules frozen",
        "check_files": ["notebooks/causal_trace_frozen_mlp_attn.ipynb"],
        "found": []
    }
}

for step, info in experiment_steps.items():
    for check_file in info["check_files"]:
        full_path = os.path.join(repo_path, check_file)
        if os.path.exists(full_path):
            info["found"].append(check_file)
    
    print(f"\n{step}")
    print(f"  Description: {info['description']}")
    print(f"  Check files: {info['check_files']}")
    print(f"  Found files: {info['found']}")
    missing = set(info['check_files']) - set(info['found'])
    if missing:
        print(f"  MISSING: {list(missing)}")
        print(f"  Status: ✗ INCOMPLETE")
    else:
        print(f"  Status: ✓ COMPLETE")

CS2 VERIFICATION: Checking Experiment Steps vs Implementation

Experiment 1: Causal Tracing of Factual Associations
  Description: Layer and token position mediators, AIE measurement
  Check files: ['experiments/causal_trace.py', 'notebooks/average_causal_effects.ipynb']
  Found files: ['experiments/causal_trace.py', 'notebooks/average_causal_effects.ipynb']
  Status: ✓ COMPLETE

Experiment 2: ROME on zsRE
  Description: ROME vs FT, FT+L, KE, MEND on zsRE 10,000 records
  Check files: ['experiments/py/eval_utils_zsre.py', 'dsets/zsre.py']
  Found files: ['experiments/py/eval_utils_zsre.py', 'dsets/zsre.py']
  Status: ✓ COMPLETE

Experiment 3: ROME Layer and Token Sweep
  Description: Target layer 0-47 and token position sweep
  Check files: ['experiments/sweep.py', 'notebooks/rome.ipynb']
  Found files: ['experiments/sweep.py', 'notebooks/rome.ipynb']
  Status: ✓ COMPLETE

Experiment 4: COUNTERFACT Evaluation
  Description: Evaluation on COUNTERFACT dataset with Score, ES, PM, NS metri

In [31]:
# Verify code functionality exists by checking key functions
print("=" * 80)
print("CS2 VERIFICATION: Checking Key Function Implementations")
print("=" * 80)

# Check causal_trace.py for key functions
causal_trace_file = os.path.join(repo_path, 'experiments', 'causal_trace.py')
with open(causal_trace_file, 'r') as f:
    causal_trace_code = f.read()

key_functions = [
    ("trace_with_patch", "Core causal tracing intervention"),
    ("calculate_hidden_flow", "Scanning all locations"),
    ("collect_embedding_std", "Computing noise level"),
    ("ModelAndTokenizer", "Model loading utility"),
]

print("\nCausal Trace Module (experiments/causal_trace.py):")
for func_name, desc in key_functions:
    if f"def {func_name}" in causal_trace_code or f"class {func_name}" in causal_trace_code:
        print(f"  ✓ {func_name}: {desc}")
    else:
        print(f"  ✗ {func_name}: {desc} - NOT FOUND")

# Check rome_main.py for key functions
rome_main_file = os.path.join(repo_path, 'rome', 'rome_main.py')
with open(rome_main_file, 'r') as f:
    rome_main_code = f.read()

rome_functions = [
    ("apply_rome_to_model", "Main ROME application function"),
    ("execute_rome", "ROME execution"),
]

print("\nROME Module (rome/rome_main.py):")
for func_name, desc in rome_functions:
    if f"def {func_name}" in rome_main_code:
        print(f"  ✓ {func_name}: {desc}")
    else:
        print(f"  ✗ {func_name}: {desc} - NOT FOUND")

CS2 VERIFICATION: Checking Key Function Implementations

Causal Trace Module (experiments/causal_trace.py):
  ✓ trace_with_patch: Core causal tracing intervention
  ✓ calculate_hidden_flow: Scanning all locations
  ✓ collect_embedding_std: Computing noise level
  ✓ ModelAndTokenizer: Model loading utility

ROME Module (rome/rome_main.py):
  ✓ apply_rome_to_model: Main ROME application function
  ✓ execute_rome: ROME execution


In [32]:
# Check compute_u.py and compute_v.py for ROME key-value computation
compute_u_file = os.path.join(repo_path, 'rome', 'compute_u.py')
compute_v_file = os.path.join(repo_path, 'rome', 'compute_v.py')

with open(compute_u_file, 'r') as f:
    compute_u_code = f.read()
with open(compute_v_file, 'r') as f:
    compute_v_code = f.read()

print("ROME Key Computation (rome/compute_u.py):")
if "def compute_u" in compute_u_code or "def get_inv_cov" in compute_u_code:
    print("  ✓ Key computation function found")
else:
    print("  ✗ Key computation function NOT FOUND")

print("\nROME Value Computation (rome/compute_v.py):")
if "def compute_v" in compute_v_code:
    print("  ✓ Value computation function found")
else:
    print("  ✗ Value computation function NOT FOUND")

# Check evaluation utils
eval_cf_file = os.path.join(repo_path, 'experiments', 'py', 'eval_utils_counterfact.py')
eval_zsre_file = os.path.join(repo_path, 'experiments', 'py', 'eval_utils_zsre.py')

with open(eval_cf_file, 'r') as f:
    eval_cf_code = f.read()
with open(eval_zsre_file, 'r') as f:
    eval_zsre_code = f.read()

print("\nEvaluation Utils (experiments/py/eval_utils_counterfact.py):")
metrics = ["rewrite_prompts_correct", "paraphrase_prompts_correct", "neighborhood_prompts_correct"]
for metric in metrics:
    if metric in eval_cf_code:
        print(f"  ✓ {metric} metric computed")
    else:
        print(f"  ✗ {metric} metric NOT FOUND")

print("\nEvaluation Utils (experiments/py/eval_utils_zsre.py):")
if "compute_rewrite_quality_zsre" in eval_zsre_code:
    print("  ✓ zsRE evaluation function found")
else:
    print("  ✗ zsRE evaluation function NOT FOUND")

ROME Key Computation (rome/compute_u.py):
  ✓ Key computation function found

ROME Value Computation (rome/compute_v.py):
  ✓ Value computation function found

Evaluation Utils (experiments/py/eval_utils_counterfact.py):
  ✗ rewrite_prompts_correct metric NOT FOUND
  ✗ paraphrase_prompts_correct metric NOT FOUND
  ✗ neighborhood_prompts_correct metric NOT FOUND

Evaluation Utils (experiments/py/eval_utils_zsre.py):
  ✓ zsRE evaluation function found


In [33]:
# Check the actual metric names in eval_utils_counterfact.py
print("=== Checking eval_utils_counterfact.py content ===")
print(eval_cf_code[:3000])

=== Checking eval_utils_counterfact.py content ===
"""
Contains evaluation utilities for pytorch-based rewriting methods.
To use, simply call `compute_rewrite_quality_counterfact` with the
appropriate arguments, which returns a dictionary containing them.
"""

import typing
from itertools import chain

import nltk
import numpy as np
import scipy
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoModelForCausalLM, AutoTokenizer

from dsets import AttributeSnippets
from util.generate import generate_fast
from util.perplexity import perplexity


def compute_rewrite_quality_counterfact(
    model: AutoModelForCausalLM,
    tok: AutoTokenizer,
    record: typing.Dict,
    snips: AttributeSnippets,
    vec: TfidfVectorizer,
) -> typing.Dict:
    """
    Given a rewritten model, computes generalization and specificity metrics for
    the desired rewrite (passed in via the CounterFact dataset record). Returns a
    dictionary containing those 

In [34]:
# Check actual metric names in the evaluation code
print("=== Checking COUNTERFACT evaluation metrics ===")
if "rewrite_prompts_probs" in eval_cf_code:
    print("  ✓ rewrite_prompts_probs (Efficacy metric)")
if "paraphrase_prompts_probs" in eval_cf_code:
    print("  ✓ paraphrase_prompts_probs (Generalization metric)")
if "neighborhood_prompts_probs" in eval_cf_code:
    print("  ✓ neighborhood_prompts_probs (Specificity metric)")
if "ngram_entropy" in eval_cf_code:
    print("  ✓ ngram_entropy (Fluency metric)")
if "reference_score" in eval_cf_code or "consistency" in eval_cf_code:
    print("  ✓ Consistency/Reference score metric")

print("\n" + "=" * 80)
print("CS2 CONCLUSION: All plan steps appear in the implementation")
print("=" * 80)

=== Checking COUNTERFACT evaluation metrics ===
  ✓ ngram_entropy (Fluency metric)
  ✓ Consistency/Reference score metric

CS2 CONCLUSION: All plan steps appear in the implementation


In [35]:
# Double check the prob metrics
print(eval_cf_code[1500:3500])

d_prompts = record["neighborhood_prompts"]
    attribute_prompts = record["attribute_prompts"]
    generation_prompts = record["generation_prompts"]

    # Form a list of lists of prefixes to test.
    prob_prompts = [
        rewrite_prompts,
        paraphrase_prompts,
        neighborhood_prompts,
        attribute_prompts,
    ]
    # Flatten all the evaluated prefixes into one list.
    probs = test_batch_prediction(
        model, tok, list(chain(*prob_prompts)), target_new["str"], target_true["str"]
    )
    # Unflatten the results again into a list of lists.
    cutoffs = [0] + np.cumsum(list(map(len, prob_prompts))).tolist()
    ret_probs = [probs[cutoffs[i - 1] : cutoffs[i]] for i in range(1, len(cutoffs))]
    # Structure the restuls as a dictionary.
    ret = {
        f"{key}_probs": ret_probs[i]
        for i, key in enumerate(
            [
                "rewrite_prompts",
                "paraphrase_prompts",
                "neighborhood_prompts",
                "a

In [36]:
# Final summary of CS2
print("=" * 80)
print("FINAL CS2 VERIFICATION SUMMARY")
print("=" * 80)

cs2_summary = """
Plan Steps and Their Implementation Status:

METHODOLOGY:
1. ✓ Causal Tracing Method: experiments/causal_trace.py implements trace_with_patch, 
   calculate_hidden_flow for causal mediation analysis
2. ✓ ROME Implementation: rome/rome_main.py, rome/compute_u.py, rome/compute_v.py 
   implement rank-one model editing
3. ✓ Evaluation Framework: experiments/evaluate.py with eval_utils for zsRE and COUNTERFACT
4. ✓ Model Support: hparams for GPT-2 XL and GPT-J configurations

EXPERIMENTS:
1. ✓ Causal Tracing: notebooks/causal_trace.ipynb, notebooks/average_causal_effects.ipynb
2. ✓ zsRE Evaluation: dsets/zsre.py, experiments/py/eval_utils_zsre.py
3. ✓ Layer/Token Sweep: experiments/sweep.py
4. ✓ COUNTERFACT Evaluation: dsets/counterfact.py, experiments/py/eval_utils_counterfact.py
5. ✓ Frozen MLP/Attn Analysis: notebooks/causal_trace_frozen_mlp_attn.ipynb

KEY FUNCTIONS VERIFIED:
- trace_with_patch (causal intervention)
- calculate_hidden_flow (AIE computation)
- apply_rome_to_model (ROME editing)
- compute_u, compute_v (key-value computation)
- compute_rewrite_quality_counterfact/zsre (evaluation)

CS2 RESULT: PASS - All plan steps appear in the implementation
"""
print(cs2_summary)

FINAL CS2 VERIFICATION SUMMARY

Plan Steps and Their Implementation Status:

METHODOLOGY:
1. ✓ Causal Tracing Method: experiments/causal_trace.py implements trace_with_patch, 
   calculate_hidden_flow for causal mediation analysis
2. ✓ ROME Implementation: rome/rome_main.py, rome/compute_u.py, rome/compute_v.py 
   implement rank-one model editing
3. ✓ Evaluation Framework: experiments/evaluate.py with eval_utils for zsRE and COUNTERFACT
4. ✓ Model Support: hparams for GPT-2 XL and GPT-J configurations

EXPERIMENTS:
1. ✓ Causal Tracing: notebooks/causal_trace.ipynb, notebooks/average_causal_effects.ipynb
2. ✓ zsRE Evaluation: dsets/zsre.py, experiments/py/eval_utils_zsre.py
3. ✓ Layer/Token Sweep: experiments/sweep.py
4. ✓ COUNTERFACT Evaluation: dsets/counterfact.py, experiments/py/eval_utils_counterfact.py
5. ✓ Frozen MLP/Attn Analysis: notebooks/causal_trace_frozen_mlp_attn.ipynb

KEY FUNCTIONS VERIFIED:
- trace_with_patch (causal intervention)
- calculate_hidden_flow (AIE computati

In [37]:
# Create the evaluation directory
eval_dir = os.path.join(repo_path, 'evaluation')
os.makedirs(eval_dir, exist_ok=True)
print(f"Created evaluation directory: {eval_dir}")

Created evaluation directory: /net/scratch2/smallyan/rome_eval/evaluation


## Summary of Consistency Evaluation

### CS1: Results vs Conclusion - PASS

All evaluable conclusions in the documentation match the results originally recorded:

| Metric | Plan Value | Documentation Value | Status |
|--------|-----------|---------------------|--------|
| Causal Tracing MLP AIE | 6.6% | 6.6% | ✓ MATCH |
| Causal Tracing Attn AIE | 1.6% | 1.6% | ✓ MATCH |
| zsRE Efficacy | 99.8% | 99.8% | ✓ MATCH |
| zsRE Paraphrase | 88.1% | 88.1% | ✓ MATCH |
| zsRE Specificity | 24.2% | 24.2% | ✓ MATCH |
| COUNTERFACT GPT-2 XL Score | 89.2 | 89.2 | ✓ MATCH |
| COUNTERFACT GPT-2 XL Efficacy | 100% | 100.0% | ✓ MATCH |
| COUNTERFACT GPT-2 XL Paraphrase | 96.4% | 96.4% | ✓ MATCH |
| COUNTERFACT GPT-2 XL Specificity | 75.4% | 75.4% | ✓ MATCH |
| COUNTERFACT GPT-J Score | 91.5 | 91.5 | ✓ MATCH |
| COUNTERFACT GPT-J Efficacy | 99.9% | 99.9% | ✓ MATCH |
| COUNTERFACT GPT-J Paraphrase | 99.1% | 99.1% | ✓ MATCH |
| COUNTERFACT GPT-J Specificity | 78.9% | 78.9% | ✓ MATCH |
| Human Eval Consistency | 1.8x | 1.8x | ✓ MATCH |
| Human Eval Fluency | 1.3x less | 1.3x less | ✓ MATCH |

### CS2: Plan vs Implementation - PASS

All plan steps are implemented in the codebase:

**Methodology Implementation:**
- ✓ Causal Tracing: `experiments/causal_trace.py`
- ✓ ROME: `rome/rome_main.py`, `rome/compute_u.py`, `rome/compute_v.py`
- ✓ Evaluation: `experiments/evaluate.py`, `experiments/py/eval_utils_*.py`
- ✓ Model Support: `hparams/ROME/gpt2-xl.json`, `hparams/ROME/EleutherAI_gpt-j-6B.json`

**Experiments Implementation:**
- ✓ Causal Tracing notebooks
- ✓ zsRE evaluation code
- ✓ Layer/Token Sweep
- ✓ COUNTERFACT evaluation
- ✓ Frozen MLP/Attn analysis

In [38]:
# Create the consistency_evaluation.json file
import json

consistency_evaluation = {
    "Checklist": {
        "CS1_Results_vs_Conclusion": "PASS",
        "CS2_Plan_vs_Implementation": "PASS"
    },
    "Rationale": {
        "CS1_Results_vs_Conclusion": "All evaluable conclusions in the documentation match the results originally recorded. Verified metrics include: Causal Tracing AIE (MLP 6.6%, Attn 1.6%), zsRE metrics (99.8% efficacy, 88.1% paraphrase, 24.2% specificity), COUNTERFACT GPT-2 XL metrics (Score 89.2, 100% efficacy, 96.4% paraphrase, 75.4% specificity), COUNTERFACT GPT-J metrics (Score 91.5, 99.9% efficacy, 99.1% paraphrase, 78.9% specificity), and Human Evaluation results (1.8x more consistent, 1.3x less fluent). All values in the plan match those in the documentation.",
        "CS2_Plan_vs_Implementation": "All plan steps appear in the implementation. The methodology includes: (1) Causal Tracing implemented in experiments/causal_trace.py with trace_with_patch and calculate_hidden_flow functions, (2) ROME implemented in rome/rome_main.py, rome/compute_u.py, rome/compute_v.py with apply_rome_to_model function, (3) Evaluation framework in experiments/evaluate.py with eval_utils for both zsRE and COUNTERFACT, (4) Model support via hparams for GPT-2 XL and GPT-J. All experiments from the plan (Causal Tracing, zsRE evaluation, Layer/Token Sweep, COUNTERFACT evaluation, Frozen MLP/Attn analysis) have corresponding implementation files."
    }
}

# Write to file
json_path = os.path.join(eval_dir, 'consistency_evaluation.json')
with open(json_path, 'w') as f:
    json.dump(consistency_evaluation, f, indent=4)

print(f"Created: {json_path}")
print("\nContent:")
print(json.dumps(consistency_evaluation, indent=4))

Created: /net/scratch2/smallyan/rome_eval/evaluation/consistency_evaluation.json

Content:
{
    "Checklist": {
        "CS1_Results_vs_Conclusion": "PASS",
        "CS2_Plan_vs_Implementation": "PASS"
    },
    "Rationale": {
        "CS1_Results_vs_Conclusion": "All evaluable conclusions in the documentation match the results originally recorded. Verified metrics include: Causal Tracing AIE (MLP 6.6%, Attn 1.6%), zsRE metrics (99.8% efficacy, 88.1% paraphrase, 24.2% specificity), COUNTERFACT GPT-2 XL metrics (Score 89.2, 100% efficacy, 96.4% paraphrase, 75.4% specificity), COUNTERFACT GPT-J metrics (Score 91.5, 99.9% efficacy, 99.1% paraphrase, 78.9% specificity), and Human Evaluation results (1.8x more consistent, 1.3x less fluent). All values in the plan match those in the documentation.",
        "CS2_Plan_vs_Implementation": "All plan steps appear in the implementation. The methodology includes: (1) Causal Tracing implemented in experiments/causal_trace.py with trace_with_patch 

In [39]:
# Copy the current notebook to the evaluation directory
import shutil

# Get the current notebook path
current_notebook = "/home/smallyan/eval_agent/notebooks/2026-01-08-15-12_ConsistencyEvaluation.ipynb"
target_notebook = os.path.join(eval_dir, 'self_matching.ipynb')

# Copy the notebook
shutil.copy(current_notebook, target_notebook)
print(f"Copied notebook to: {target_notebook}")

# Verify both files exist
print("\nEvaluation directory contents:")
for f in os.listdir(eval_dir):
    print(f"  - {f}")

Copied notebook to: /net/scratch2/smallyan/rome_eval/evaluation/self_matching.ipynb

Evaluation directory contents:
  - consistency_evaluation.json
  - self_matching.ipynb


## Final Binary Checklist

| Checklist Item | Result |
|----------------|--------|
| **CS1. Conclusion vs Original Results** | **PASS** |
| **CS2. Implementation Follows the Plan** | **PASS** |

### Files Created:
1. `/net/scratch2/smallyan/rome_eval/evaluation/self_matching.ipynb` - This notebook with the evaluation analysis
2. `/net/scratch2/smallyan/rome_eval/evaluation/consistency_evaluation.json` - JSON summary of results

In [40]:
# Update the notebook copy with the final content
shutil.copy(current_notebook, target_notebook)
print(f"Updated notebook at: {target_notebook}")

# Verify the JSON file content
with open(json_path, 'r') as f:
    verification = json.load(f)
print("\nFinal consistency_evaluation.json verification:")
print(json.dumps(verification, indent=2))

Updated notebook at: /net/scratch2/smallyan/rome_eval/evaluation/self_matching.ipynb

Final consistency_evaluation.json verification:
{
  "Checklist": {
    "CS1_Results_vs_Conclusion": "PASS",
    "CS2_Plan_vs_Implementation": "PASS"
  },
  "Rationale": {
    "CS1_Results_vs_Conclusion": "All evaluable conclusions in the documentation match the results originally recorded. Verified metrics include: Causal Tracing AIE (MLP 6.6%, Attn 1.6%), zsRE metrics (99.8% efficacy, 88.1% paraphrase, 24.2% specificity), COUNTERFACT GPT-2 XL metrics (Score 89.2, 100% efficacy, 96.4% paraphrase, 75.4% specificity), COUNTERFACT GPT-J metrics (Score 91.5, 99.9% efficacy, 99.1% paraphrase, 78.9% specificity), and Human Evaluation results (1.8x more consistent, 1.3x less fluent). All values in the plan match those in the documentation.",
    "CS2_Plan_vs_Implementation": "All plan steps appear in the implementation. The methodology includes: (1) Causal Tracing implemented in experiments/causal_trace.py w

In [41]:
# Final copy of the notebook
shutil.copy(current_notebook, target_notebook)
print("=" * 80)
print("CONSISTENCY EVALUATION COMPLETE")
print("=" * 80)
print(f"\nOutput files created at: {eval_dir}")
print(f"  1. self_matching.ipynb - Full evaluation notebook")
print(f"  2. consistency_evaluation.json - Binary checklist results")
print("\nResults:")
print("  CS1 (Results vs Conclusion): PASS")
print("  CS2 (Plan vs Implementation): PASS")